In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
# Download the dataset (using a common PlantVillage version)
!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip

In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# Transformations for Research Accuracy
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # Normalization based on ImageNet stats (Standard for ResNet)
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load data - Point this to the 'color' folder inside the unzipped directory
dataset = datasets.ImageFolder(root='plantvillage dataset/color', transform=data_transforms)

# Split into Train (80%) and Test (20%)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pre-trained ResNet18
model = models.resnet18(weights='IMAGENET1K_V1')

# Change the last layer to match the number of plant classes (usually 38)
num_classes = len(dataset.classes)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)
print(f"Model loaded and sent to {device}")

In [ ]:
import torch.optim as optim
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop (Run for 5-10 epochs for a strong baseline)
for epoch in range(5):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1} - Loss: {running_loss/len(train_loader):.4f}")

In [ ]:
torch.save(model.state_dict(), 'plant_resnet18_baseline.pth')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

def evaluate_and_plot(model, loader, class_names):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate Accuracy
    correct = (np.array(all_preds) == np.array(all_labels)).sum()
    total = len(all_labels)
    print(f"Final Test Accuracy: {(correct/total)*100:.2f}%")

    # 1. Classification Report (Precision, Recall, F1-Score for your paper)
    print("\nClassification Report:\n")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # 2. Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(15, 12))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix: Baseline ResNet18')
    plt.show()

# Run it!
evaluate_and_plot(model, test_loader, dataset.classes)

In [ ]:
torch.save(model.state_dict(), 'resnet18_plantvillage_final.pth')
print("Model saved successfully!")


In [ ]:
!pip install adversarial-robustness-toolbox

In [ ]:
# Updated imports for ART 1.14+
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod  # Note the slight name change in newer versions

# 1. Wrap your model in an ART interface
classifier = PyTorchClassifier(
    model=model,
    clip_values=(0, 1),
    loss=criterion,
    optimizer=optimizer,
    input_shape=(3, 224, 224),
    nb_classes=num_classes,
)

# 2. Initialize the FGSM Attack (using the updated class name)
# We set 'targeted=False' because we just want the AI to be WRONG,
# we don't care which wrong class it chooses.
attack = FastGradientMethod(estimator=classifier, eps=0.1, targeted=False)

# 3. Create a small batch of adversarial images for testing
images, labels = next(iter(test_loader))
images_numpy = images.cpu().numpy()

# Generate the Adversarial Examples
x_test_adv = attack.generate(x=images_numpy)

print("Adversarial images generated successfully!")

In [ ]:
import matplotlib.pyplot as plt

def plot_attack(original, adversarial, label, target_model):
    # Prepare images for display (un-normalize or clip to 0-1)
    org_img = np.transpose(original, (1, 2, 0))
    adv_img = np.transpose(adversarial, (1, 2, 0))
    noise = adv_img - org_img

    # Get Predictions
    org_preds = target_model(torch.from_numpy(original).unsqueeze(0).to(device))
    adv_preds = target_model(torch.from_numpy(adversarial).unsqueeze(0).to(device))

    org_class = dataset.classes[torch.argmax(org_preds).item()]
    adv_class = dataset.classes[torch.argmax(adv_preds).item()]

    # Plotting
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.title(f"Original: {org_class}")
    plt.imshow(np.clip(org_img, 0, 1))
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.title("Adversarial Noise (Perturbation)")
    # We magnify the noise so it's visible to human eyes
    plt.imshow(np.clip(noise * 10, 0, 1))
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.title(f"Attacked: {adv_class}")
    plt.imshow(np.clip(adv_img, 0, 1))
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# Visualize the first image in the batch
plot_attack(images_numpy[0], x_test_adv[0], labels[0], model)

In [ ]:
def check_adversarial_accuracy(classifier, loader, attack_engine):
    correct_clean = 0
    correct_adv = 0
    total = 0

    # We test on the first 100 images to save time (increase for final paper)
    for i, (images, labels) in enumerate(loader):
        if i > 3: break # Test on ~128 images for a quick check

        images_np = images.numpy()

        # 1. Predict on Clean
        preds_clean = classifier.predict(images_np)
        correct_clean += np.sum(np.argmax(preds_clean, axis=1) == labels.numpy())

        # 2. Generate Attack & Predict
        x_adv = attack_engine.generate(x=images_np)
        preds_adv = classifier.predict(x_adv)
        correct_adv += np.sum(np.argmax(preds_adv, axis=1) == labels.numpy())

        total += len(labels)

    print(f"Clean Accuracy: {(correct_clean/total)*100:.2f}%")
    print(f"Adversarial Accuracy: {(correct_adv/total)*100:.2f}%")

check_adversarial_accuracy(classifier, test_loader, attack)

In [ ]:
import torch.nn as nn

class LeafShield(nn.Module):
    def __init__(self):
        super(LeafShield, self).__init__()
        # Encoder: Shrinks the image to find core features
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1), # 112x112
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), # 56x56
            nn.ReLU()
        )
        # Decoder: Rebuilds the image, leaving the noise behind
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 3, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid() # Keeps pixel values between 0 and 1
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

# Initialize the Shield
shield = LeafShield().to(device)
print("Shield Layer initialized.")


In [ ]:
import torch.optim as optim

shield_optimizer = optim.Adam(shield.parameters(), lr=0.001)
shield_criterion = nn.MSELoss() # We use Mean Squared Error to compare pixels

# Quick training loop for the Shield (3-5 Epochs)
for epoch in range(3):
    for images, _ in train_loader:
        images = images.to(device)

        # Add random noise to simulate an attack
        noisy_images = images + 0.1 * torch.randn_like(images)
        noisy_images = torch.clamp(noisy_images, 0, 1)

        # Train Shield to clean the noise
        shield_optimizer.zero_grad()
        cleaned_images = shield(noisy_images)
        loss = shield_criterion(cleaned_images, images)
        loss.backward()
        shield_optimizer.step()

    print(f"Shield Training Epoch {epoch+1} - Recovery Loss: {loss.item():.4f}")


In [ ]:
def final_defense_evaluation(classifier, shield, loader, attack):
    model.eval()
    shield.eval()

    correct_clean = 0
    correct_adv = 0
    correct_defended = 0
    total = 0

    print("Running Final Defense Evaluation... (this may take a minute)")

    # We REMOVED torch.no_grad() here so the attack can compute gradients
    for i, (images, labels) in enumerate(loader):
        if i > 5: break # Sample ~192 images

        images_v = images.to(device)
        labels_v = labels.to(device)

        # 1. Clean Accuracy (No grad needed here, but keeping it simple)
        outputs_clean = model(images_v)
        correct_clean += (torch.argmax(outputs_clean, 1) == labels_v).sum().item()

        # 2. Adversarial Accuracy (The Breach)
        # ART generates the attack - it handles its own gradient tracking internally
        x_adv_np = attack.generate(x=images.numpy())
        x_adv_v = torch.from_numpy(x_adv_np).to(device)

        outputs_adv = model(x_adv_v)
        correct_adv += (torch.argmax(outputs_adv, 1) == labels_v).sum().item()

        # 3. Defended Accuracy (The Rescue)
        # We pass the attacked image through the shield
        x_defended_v = shield(x_adv_v)
        outputs_defended = model(x_defended_v)
        correct_defended += (torch.argmax(outputs_defended, 1) == labels_v).sum().item()

        total += labels.size(0)

    print("-" * 30)
    print(f"RESULTS FOR RESEARCH PAPER:")
    print(f"Clean Accuracy: {(correct_clean/total)*100:.2f}%")
    print(f"Adversarial Accuracy (No Defense): {(correct_adv/total)*100:.2f}%")
    print(f"Defended Accuracy (With LeafShield): {(correct_defended/total)*100:.2f}%")
    print("-" * 30)

# Run the fixed evaluation
final_defense_evaluation(classifier, shield, test_loader, attack)


In [ ]:
import torch.nn.functional as F
from scipy.ndimage import median_filter

def final_defense_evaluation_v2(classifier, loader, attack):
    model.eval()

    correct_clean = 0
    correct_adv = 0
    correct_defended = 0
    total = 0

    print("Running Revised Defense Evaluation (Median Filtering)...")

    for i, (images, labels) in enumerate(loader):
        if i > 5: break

        images_v = images.to(device)
        labels_v = labels.to(device)

        # 1. Clean
        outputs_clean = model(images_v)
        correct_clean += (torch.argmax(outputs_clean, 1) == labels_v).sum().item()

        # 2. Adversarial
        x_adv_np = attack.generate(x=images.numpy())
        x_adv_v = torch.from_numpy(x_adv_np).to(device)
        outputs_adv = model(x_adv_v)
        correct_adv += (torch.argmax(outputs_adv, 1) == labels_v).sum().item()

        # 3. New Defense: Median Filtering (Denoising without a Neural Network)
        # This is a standard 'Non-Differentiable' defense in security research
        x_def_np = x_adv_np.copy()
        for b in range(x_def_np.shape[0]):
            for c in range(3):
                # We apply a 3x3 median filter to 'pop' the adversarial pixels
                x_def_np[b, c] = median_filter(x_def_np[b, c], size=3)

        x_def_v = torch.from_numpy(x_def_np).to(device)
        outputs_defended = model(x_def_v)
        correct_defended += (torch.argmax(outputs_defended, 1) == labels_v).sum().item()

        total += labels.size(0)

    print("-" * 30)
    print(f"REVISED RESULTS:")
    print(f"Clean Accuracy: {(correct_clean/total)*100:.2f}%")
    print(f"Adversarial Accuracy: {(correct_adv/total)*100:.2f}%")
    print(f"Defended Accuracy (Median Filter): {(correct_defended/total)*100:.2f}%")
    print("-" * 30)

final_defense_evaluation_v2(classifier, test_loader, attack)


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. We use only Resize and ToTensor to stay in the [0, 1] range
hardened_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# 2. Re-load the dataset (Ensure the folder name is correct)
DATA_PATH = 'plantvillage dataset/color'
full_dataset = datasets.ImageFolder(root=DATA_PATH, transform=hardened_transforms)

torch.manual_seed(42)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Data ready. Training on: {len(train_dataset)} images.")

In [ ]:
import torch.optim as optim
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod

# 1. Setup the ART wrapper
# We use a lower learning rate (0.0001) to keep the 'brain surgery' stable
classifier = PyTorchClassifier(
    model=model,
    clip_values=(0, 1),
    loss=torch.nn.CrossEntropyLoss(),
    optimizer=optim.Adam(model.parameters(), lr=0.0001),
    input_shape=(3, 224, 224),
    nb_classes=38,
)

# 2. Setup the training attack
train_attack = FastGradientMethod(estimator=classifier, eps=0.1)

print("Starting Hybrid-Hardening Rescue Loop...")

model.train()
for epoch in range(2): # 2 epochs is enough for a massive robustness jump
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images_v, labels_v = images.to(device), labels.to(device)

        # Generate adversarial images for half of the batch
        images_np = images.numpy()
        x_adv = train_attack.generate(x=images_np)
        x_adv_v = torch.from_numpy(x_adv).to(device)

        # Mix them: 16 clean, 16 adversarial
        combined_images = torch.cat([images_v[:16], x_adv_v[16:]], dim=0)

        classifier.optimizer.zero_grad()
        outputs = model(combined_images)
        loss = torch.nn.CrossEntropyLoss()(outputs, labels_v)
        loss.backward()
        classifier.optimizer.step()

        running_loss += loss.item()
        if i % 100 == 0:
            print(f"Batch {i} - Training Loss: {loss.item():.4f}")

    print(f"Epoch {epoch+1} Complete. Avg Loss: {running_loss/len(train_loader):.4f}")

print("Rescue Complete! Your model is now Hardened.")

In [ ]:
def final_robustness_audit(hardened_model, loader, attack_engine):
    hardened_model.eval()
    correct_clean = 0
    correct_adv = 0
    total = 0

    print("Running Final Audit on Hardened Model...")

    # Gradients are needed for the attack_engine to generate perturbations
    for i, (images, labels) in enumerate(loader):
        if i > 20: break # Testing on ~670 images for high statistical confidence

        images_v, labels_v = images.to(device), labels.to(device)

        # 1. Clean Accuracy (Standard Test)
        outputs_clean = hardened_model(images_v)
        correct_clean += (torch.argmax(outputs_clean, 1) == labels_v).sum().item()

        # 2. Adversarial Accuracy (The 'Breach' Test)
        # We generate the attack images based on the clean ones
        x_adv_np = attack_engine.generate(x=images.numpy())
        x_adv_v = torch.from_numpy(x_adv_np).to(device)

        outputs_adv = hardened_model(x_adv_v)
        correct_adv += (torch.argmax(outputs_adv, 1) == labels_v).sum().item()

        total += labels.size(0)

    print("\n" + "="*40)
    print("      FINAL PUBLISHABLE RESULTS        ")
    print("="*40)
    print(f"Hardened Clean Accuracy:      {(correct_clean/total)*100:.2f}%")
    print(f"Hardened Adversarial Accuracy: {(correct_adv/total)*100:.2f}%")
    print("="*40)

    # Save the successful model
    torch.save(hardened_model.state_dict(), 'leaf_model_hardened_final.pth')
    print("Final Model Saved: leaf_model_hardened_final.pth")

# Run the final check
final_robustness_audit(model, test_loader, train_attack)

In [ ]:
from art.attacks.evasion import ProjectedGradientDescent

# 1. Setup a stronger attack for final 'Deep Hardening'
# eps_step is the size of each mini-step; max_iter is the number of steps
pgd_attack = ProjectedGradientDescent(
    estimator=classifier,
    eps=0.1,
    eps_step=0.01,
    max_iter=7,
    batch_size=32
)

print("Starting Deep Hardening with PGD (Targeting 90%)...")
print("Note: This will be ~7x slower than the previous loop.")

model.train()
running_loss = 0.0
for i, (images, labels) in enumerate(train_loader):
    if i > 500: break # We only need a 'finishing touch' on about 16,000 images

    images_v, labels_v = images.to(device), labels.to(device)

    # Generate the strongest possible attack (PGD)
    x_adv = pgd_attack.generate(x=images.numpy())
    x_adv_v = torch.from_numpy(x_adv).to(device)

    # Hybrid Training
    combined_images = torch.cat([images_v[:16], x_adv_v[16:]], dim=0)

    classifier.optimizer.zero_grad()
    outputs = model(combined_images)
    loss = torch.nn.CrossEntropyLoss()(outputs, labels_v)
    loss.backward()
    classifier.optimizer.step()

    if i % 20 == 0:
        print(f"Batch {i} - PGD Loss: {loss.item():.4f}")

print("Deep Hardening Complete!")

In [ ]:
# 1. Load the successful 78% model weights
model.load_state_dict(torch.load('leaf_model_hardened_final.pth'))
model = model.to(device)
model.eval()

# 2. Re-verify the "Sweet Spot" results
def verify_success(hardened_model, loader):
    correct_clean = 0
    correct_adv = 0
    total = 0

    # Use the same FGSM attack settings
    verify_attack = FastGradientMethod(estimator=classifier, eps=0.1)

    print("🔄 Restoring the 78% Breakthrough Model...")

    # We REMOVE torch.no_grad() here so ART can work
    for i, (images, labels) in enumerate(loader):
        if i > 10: break
        images_v, labels_v = images.to(device), labels.to(device)

        # Clean Prediction
        out_c = hardened_model(images_v)
        correct_clean += (torch.argmax(out_c, 1) == labels_v).sum().item()

        # Adversarial Prediction (ART needs gradients for this part)
        x_adv = verify_attack.generate(x=images.numpy())
        x_adv_v = torch.from_numpy(x_adv).to(device)
        out_a = hardened_model(x_adv_v)
        correct_adv += (torch.argmax(out_a, 1) == labels_v).sum().item()

        total += labels.size(0)

    print(f"\n✅ Restore Complete!")
    print(f"Clean Accuracy restored to: {(correct_clean/total)*100:.2f}%")
    print(f"Robust Accuracy restored to: {(correct_adv/total)*100:.2f}%")

verify_success(model, test_loader)

In [ ]:
import torch.optim as optim

# 1. Ensure we are starting from our best saved version
model.load_state_dict(torch.load('leaf_model_hardened_final.pth'))
model.to(device)

# 2. Use a VERY small learning rate to avoid the "collapse" we saw before
optimizer = optim.Adam(model.parameters(), lr=0.00001)

# 3. Re-initialize the attack
refine_attack = FastGradientMethod(estimator=classifier, eps=0.1)

print("Starting Final Polish (Stabilizing Robustness)...")

model.train()
for i, (images, labels) in enumerate(train_loader):
    if i > 200: break # Only a quick 5-minute polish

    images_v, labels_v = images.to(device), labels.to(device)

    # Generate attack
    x_adv = refine_attack.generate(x=images.numpy())
    x_adv_v = torch.from_numpy(x_adv).to(device)

    # Standard Hybrid Training
    combined_images = torch.cat([images_v[:16], x_adv_v[16:]], dim=0)

    optimizer.zero_grad()
    outputs = model(combined_images)
    loss = torch.nn.CrossEntropyLoss()(outputs, labels_v)
    loss.backward()
    optimizer.step()

print("Polish Complete! Running Final Final Audit...")

# 4. Final Audit (No no_grad block for the attack part)
model.eval()
c_acc, r_acc, total = 0, 0, 0
for i, (images, labels) in enumerate(test_loader):
    if i > 15: break
    img_v, lab_v = images.to(device), labels.to(device)

    # Clean
    c_acc += (torch.argmax(model(img_v), 1) == lab_v).sum().item()
    # Robust
    x_adv = refine_attack.generate(x=images.numpy())
    r_acc += (torch.argmax(model(torch.from_numpy(x_adv).to(device)), 1) == lab_v).sum().item()
    total += labels.size(0)

print(f"\n✨ FINAL STABILIZED RESULTS ✨")
print(f"Clean Accuracy: {(c_acc/total)*100:.2f}%")
print(f"Robust Accuracy: {(r_acc/total)*100:.2f}%")